In [ ]:
import sys, os

sys.path.append(os.path.dirname(os.getcwd()))

import h5py
import tqdm
import matplotlib.pyplot as plt

plt.rcParams.update({"mathtext.fontset": "cm"})

import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from flax import nnx
import optax

import models.time_solver as imex
from models.sph_mhd import (
    SphericalMHD,
    dynamical_solver,
    integral,
    kinetic_energy,
)

In [ ]:
eq = SphericalMHD(
    Ra=80, E=1e-3, Pr=1, radius_ratio=0.35, l_max=31, m_max=7, m_res=4, n_r=84
)
eq

In [ ]:
with h5py.File("../data/calibration/calibration_samples.h5", "r") as f:
    dt = f.attrs["dt"]
    steps = f.attrs["steps"]

    t_0 = np.array(f["t_0"][:])
    # (samples, features, r, lm) -> (samples, r, lm, features)
    f_m = np.moveaxis(np.array(f["f_m"][:]), 1, -1)

f_means = np.mean(f_m.real, axis=(0, 1, 2)) + 1j * np.mean(f_m.imag, axis=(0, 1, 2))
f_stds = np.std(f_m.real, axis=(0, 1, 2)) + 1j * np.std(f_m.imag, axis=(0, 1, 2))

print(
    """Dataset statistics: 
        theta_lm = {:.4} ± σ({:.4})
        u_tor_lm = {:.4} ± σ({:.4})
        u_pol_lm = {:.4} ± σ({:.4})""".format(
        f_means[0],
        f_stds[0],
        f_means[1],
        f_stds[1],
        f_means[2],
        f_stds[2],
    )
)

sub_trajs = int(len(f_m) / steps)
f_m = np.array(np.split(f_m, sub_trajs, axis=0))

seed = 42
key = jax.random.key(seed)


def mod_relu(z: jnp.ndarray, b: float = 1.0, c: float = 1e-3) -> jnp.ndarray:
    abs_z = jnp.abs(z)
    return nnx.relu(abs_z + b) * z / (abs_z + c)


class NextBlock(nnx.Module):
    def __init__(
        self,
        activation,
        in_features: int,
        out_features: int,
        kernel_size,
        rngs,
    ):
        self.features = out_features
        self.conv = nnx.Conv(in_features, out_features, kernel_size, rngs=rngs)
        self.lin_1 = nnx.Linear(out_features, out_features, rngs=rngs)
        self.act = activation
        self.lin_2 = nnx.Linear(out_features, out_features, rngs=rngs)

    def __call__(self, x):
        x = self.conv(x)
        x_id = x
        x = self.lin_1(x)
        x = self.act(x)
        x = self.lin_2(x)
        x += x_id
        return x


class SphConvNext(nnx.Module):
    def __init__(self, blocks, means: jnp.ndarray, stds: jnp.ndarray, activation, rngs):
        self.means = nnx.Variable(means)
        self.stds = nnx.Variable(stds)
        self.blocks = nnx.List()
        cur_features_U = 2
        cur_features_T = 1
        for kernel, features in blocks:
            self.blocks.append(
                nnx.data(
                    (
                        NextBlock(
                            activation,
                            cur_features_U,
                            features,
                            (kernel, kernel),
                            rngs=rngs,
                        ),  # U block
                        NextBlock(
                            activation,
                            cur_features_T,
                            features,
                            (kernel, kernel),
                            rngs=rngs,
                        ),  # om_m block
                    )
                )
            )
            cur_features_U = cur_features_T = features
        self.lin_U = nnx.Linear(cur_features_U, 2, rngs=rngs)
        self.lin_T = nnx.Linear(cur_features_T, 1, rngs=rngs)

    def __call__(self, f):
        f_real = (f.real - self.means.real) / self.stds.real
        f_imag = (f.imag - self.means.imag) / self.stds.imag
        f = f_real + 1j * f_imag
        U = f[..., 1:]
        T = f[..., :1]
        for U_block, T_block in self.blocks:
            U = U_block(U)
            T = T_block(T)
        U = self.lin_U(U)
        T = self.lin_T(T)
        return jnp.concatenate((U, T), axis=-1)


model = SphConvNext(
    blocks=[(3, 16), (3, 32), (3, 64)],
    means=f_means,
    stds=f_stds,
    activation=mod_relu,
    rngs=nnx.Rngs(key),
)
# print(model)

optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)


def kinetic_energy_loss(
    f_m_hat: jnp.ndarray,
    f_m: jnp.ndarray,
) -> np.ndarray:
    (
        _,
        u_tor_lm_hat,
        u_pol_lm_hat,
    ) = jnp.moveaxis(f_m_hat, -1, 0)
    (
        _,
        u_tor_lm,
        u_pol_lm,
    ) = jnp.moveaxis(f_m, -1, 0)

    l2 = eq.l_map * (eq.l_map + 1)

    u_tor_mis_lm = u_tor_lm_hat - u_tor_lm
    u_pol_mis_lm = u_pol_lm_hat - u_pol_lm

    eke_lm = kinetic_energy(eq, u_tor_mis_lm, u_pol_mis_lm)
    return jnp.sum(eke_lm[:, :, eq.sh.m != 0]) + jnp.sum(
        0.5 * eke_lm[:, :, eq.sh.m == 0]
    )


def flow_loss(
    model: nnx.Module, t_0: jnp.ndarray, f_0: jnp.ndarray, f_m: jnp.ndarray
) -> float:
    def source(theta_lm, u_tor_lm, u_pol_lm):
        tau_lm = model(
            jnp.expand_dims(jnp.stack((theta_lm, u_tor_lm, u_pol_lm), axis=-1), 0)
        ).squeeze()
        return jnp.moveaxis(tau_lm, -1, 0)

    solver = dynamical_solver(
        eq,
        imex.PC2(dt),
        source,
    )

    def __loop__(states: jnp.ndarray, cur_step: int) -> jnp.ndarray:
        theta_lm, u_tor_lm, u_pol_lm = states
        c, *states = solver(theta_lm, u_tor_lm, u_pol_lm)
        return (jnp.array(states), jnp.stack(states, axis=-1))

    _, f_m_hat = jax.lax.scan(__loop__, jnp.moveaxis(f_0, -1, 0), length=steps - 1)
    return kinetic_energy_loss(
        f_m_hat,
        f_m,
    )


@nnx.jit
def train_step(model, optimizer, traj_batch: jnp.ndarray):
    t_0, f_0, f_m = traj_batch
    loss, grads = nnx.value_and_grad(flow_loss)(model, t_0, f_0, f_m)
    optimizer.update(model, grads)
    return loss


# Batch generator
def batch_gen(t_0: np.ndarray, f_m: np.ndarray, sub_trajs: int, shuffle=True, key=None):
    """Generate trajectories (batch of continous samples) from a given dataset."""
    idx = np.arange(sub_trajs)
    if shuffle:
        idx = jax.random.permutation(key, idx, independent=True)

    for batch in range(sub_trajs):
        curr_idx = idx[batch]
        batch_inputs = f_m[curr_idx, 0]
        batch_target = f_m[curr_idx, 1:]
        yield (t_0[curr_idx], jnp.array(batch_inputs), jnp.array(batch_target))


train_loss = []
print("Training subgrid-scale model...")
pbar = tqdm.tqdm(range(25), bar_format="{l_bar}{bar:10}{r_bar}{bar:-10b}")
for i in pbar:
    key, subkey = jax.random.split(key)
    data_sample = batch_gen(t_0, f_m, sub_trajs, shuffle=True, key=subkey)

    t_loss = 0.0
    for traj in range(sub_trajs):
        traj_batch = next(data_sample)
        loss = train_step(model, optimizer, traj_batch)
        t_loss += loss / (steps - 1)

        pbar.set_postfix(
            sub_traj=traj,
            loss=t_loss / (traj + 1),
        )
    train_loss.append(t_loss / sub_trajs)

In [ ]:
fig, axs = plt.subplots(figsize=(3, 3), ncols=1)

axs.plot(train_loss)
axs.set_ylabel("Train loss")
axs.set_xlabel("Epoch")

fig.tight_layout()
plt.show()